In [1]:
#!/usr/bin/env python3
import re

input_file = "night_yp0901.txt"   # 输入文件
output_file = "night_yp0901.c"  # 输出文件

# 匹配 rgb_tuple_from(0,0,0) 这种格式
pattern = re.compile(r'rgb_tuple_from\s*\(\s*(\d+)\s*,\s*(\d+)\s*,\s*(\d+)\s*\)')

colors = []

with open(input_file, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        match = pattern.search(line)
        if match:
            r, g, b = map(int, match.groups())
            colors.append((r, g, b))

num = len(colors)

with open(output_file, "w") as f:
    f.write(f"uint8_t colors[{num}][3] = {{\n")
    for r, g, b in colors:
        f.write(f"    {{0x{r:02X}, 0x{g:02X}, 0x{b:02X}}},\n")
    f.write("};\n")

print(f"Parsed {num} colors into {output_file}")

Parsed 256 colors into night_yp0901.c


In [2]:
import cv2

colormaps = [name for name in dir(cv2) if name.startswith("COLORMAP_")]
print("OpenCV Colormaps:")
for name in colormaps:
    print(name)


OpenCV Colormaps:
COLORMAP_AUTUMN
COLORMAP_BONE
COLORMAP_CIVIDIS
COLORMAP_COOL
COLORMAP_DEEPGREEN
COLORMAP_HOT
COLORMAP_HSV
COLORMAP_INFERNO
COLORMAP_JET
COLORMAP_MAGMA
COLORMAP_OCEAN
COLORMAP_PARULA
COLORMAP_PINK
COLORMAP_PLASMA
COLORMAP_RAINBOW
COLORMAP_SPRING
COLORMAP_SUMMER
COLORMAP_TURBO
COLORMAP_TWILIGHT
COLORMAP_TWILIGHT_SHIFTED
COLORMAP_VIRIDIS
COLORMAP_WINTER


In [ ]:
import cv2
import numpy as np

# 选择 colormap
colormap_name = "COLORMAP_HOT"
colormap_dict = {
    name: getattr(cv2, name) for name in dir(cv2) if name.startswith("COLORMAP_")
}
colormap = colormap_dict[colormap_name]

# 生成 0~255 灰度
gray = np.arange(256, dtype=np.uint8).reshape(1, 256)

# 应用 OpenCV colormap
color = cv2.applyColorMap(gray, colormap)  # 输出 BGR

# 转成 RGB
rgb = color[0, :, ::-1]  # shape (256, 3)

# 写入 C 文件
c_filename = f"lut_{colormap_name.lower()}.c"
with open(c_filename, "w") as f:
    f.write(f"#include <stdint.h>\n\n")
    f.write(f"const uint8_t {colormap_name.lower()}[256][3] = {{\n")
    for i in range(256):
        r, g, b = int(rgb[i,0]), int(rgb[i,1]), int(rgb[i,2])
        f.write(f"    {{0x{r:02X}, 0x{g:02X}, 0x{b:02X}}},\n")
    f.write("};\n")

print(f"C LUT file generated: {c_filename}")


C LUT file generated: lut_colormap_hot.c


: 

In [ ]:
import cv2
import numpy as np

# 选择 colormap
colormap_name = "COLORMAP_HOT"
colormap_dict = {
    name: getattr(cv2, name) for name in dir(cv2) if name.startswith("COLORMAP_")
}
colormap = colormap_dict[colormap_name]

# 生成 0~255 灰度
gray = np.arange(256, dtype=np.uint8).reshape(1, 256)

# 应用 OpenCV colormap（输出 BGR）
color = cv2.applyColorMap(gray, colormap)

# 转成 RGB，shape = (256, 3)
rgb = color[0, :, ::-1]

# 写入 bin 文件（严格 256*3 = 768 字节）
bin_filename = f"lut_{colormap_name.lower()}.bin"
with open(bin_filename, "wb") as f:
    f.write(rgb.tobytes())

print(f"Binary LUT file generated: {bin_filename}, size={rgb.nbytes} bytes")

Binary LUT file generated: lut_colormap_hot.bin, size=768 bytes


: 